In [21]:
!pip install gurobipy

In [22]:
import numpy as np
import cvxpy as cp

def solve_master_problem(I, J, f_j, d_ij, big_lambda_i, cuts):

  # Decision variables
  x_j = cp.Variable(J, boolean=True)
  y_ij = cp.Variable((I, J), nonneg=True)
  overline_lambda_j = cp.Variable(J, nonneg=True)
  theta_j = cp.Variable(J)  # recourse can be negative when revenue dominates

  # Unit travel cost times zone demand: c_ij = d_ij * Lambda_i
  travel_cost = d_ij * big_lambda_i[:, np.newaxis]

  # Objective and constraints
  objective = cp.Minimize(
      cp.sum(cp.multiply(f_j, x_j))
      + cp.sum(cp.multiply(travel_cost, y_ij))
      + cp.sum(theta_j)
  )

  constraints = [
      cp.sum(y_ij, axis=1) == 1,
      y_ij <= x_j,
      overline_lambda_j == y_ij.T @ big_lambda_i,
      y_ij <= 1,
  ]
  if len(cuts) == 0:
      constraints.append(theta_j >= 0)

  # Integer-lifted optimality cuts: theta_j >= alpha * x_j + pi * lambda_j
  for station, alpha, pi in cuts:
    constraints.append(
        theta_j[station] >= alpha * x_j[station] + pi * overline_lambda_j[station]
    )

  prob = cp.Problem(objective, constraints)

  result = prob.solve(solver=cp.GUROBI)
  return result, overline_lambda_j.value, x_j.value, y_ij.value

In [23]:
def solve_subproblem(j, kappa_j, v_j, p, x_j_val, M_j_val, lam_bar_j_val,
                     epsilon=2, n_tangent=100):
  """Station-level recourse subproblem (SP-j) for a single open/closed station."""

  lam = cp.Variable(nonneg=True)
  s = cp.Variable(nonneg=True)
  tau = cp.Variable(nonneg=True)
  z = cp.Variable(nonneg=True)

  def phi(t):
      return t / (1 - np.sqrt(t))

  def dphi(t):
      return (2 - np.sqrt(t)) / (2 * (1 - np.sqrt(t))**2)

  price = float(np.asarray(p).ravel()[0])

  objective = cp.Minimize(
      kappa_j * s + v_j * z - price * lam
  )
  dual_constraint = lam <= lam_bar_j_val

  constraints = [
      dual_constraint,
      cp.square(lam + epsilon * x_j_val) <= s,
      s <= M_j_val**2 * x_j_val,
      tau <= 0.999,
      cp.quad_over_lin(lam, s) <= tau,
  ]

  grid = np.linspace(0.05, 0.95, n_tangent)
  for t in grid:
      constraints.append(z >= phi(t) + dphi(t) * (tau - t))

  prob = cp.Problem(objective, constraints)
  result = prob.solve()
  pi_j = float(dual_constraint.dual_value)
  return result, pi_j

In [24]:
def OA_GBD_solve(I, J, f_j, d_ij, big_lambda_i,
                 tol, kappa, v_j, M_j, p, max_iter=50, verbose=True):

    LB = -np.inf
    UB = np.inf
    k = 0
    cuts = []

    while k < max_iter and (k == 0 or abs(UB - LB) > tol * max(1, abs(UB))):
        master_obj, lambda_soln, x_j_soln, y_ij_soln = \
            solve_master_problem(I, J, f_j, d_ij, big_lambda_i, cuts)

        LB = master_obj

        incumbent = (
            np.sum(f_j * x_j_soln)
            + np.sum(d_ij * big_lambda_i[:, None] * y_ij_soln)
        )

        for j in range(J):
            recourse_value, pi_j = solve_subproblem(
                j, kappa[j], v_j[j], p,
                x_j_soln[j], M_j[j], lambda_soln[j]
            )

            alpha = float(recourse_value - pi_j * lambda_soln[j])
            incumbent += recourse_value
            cuts.append((j, alpha, pi_j))
            if verbose:
                print(f"  station {j}: pi={pi_j:.6f}, recourse={recourse_value:.4f}")

        UB = min(UB, incumbent)
        k += 1
        if verbose:
            gap = UB - LB
            print(f"Iteration {k}: LB={LB:.4f}, UB={UB:.4f}, gap={gap:.4f}")

    if verbose:
        print(f"Finished in {k} iterations. Best incumbent (UB) = {UB:.4f}")
    return UB, LB, x_j_soln, y_ij_soln, lambda_soln, cuts

In [25]:
# Sets
I = 1
J = 2
# Parameters
f_j = np.array([80, 100])
d_ij = np.array([[100], [15]])
big_lambda_i = np.array([15.0])
tol = 0.001
kappa_j = np.array([40, 50])
v_j = np.array([0.8, 0.7])
p = np.array([10]) # Assumes linear revenue r(lambda) = p * lambda

M_j = np.array([40.0, 15.0])

OA_GBD_solve(I, J, f_j, d_ij, big_lambda_i,
                 tol, kappa_j, v_j, M_j, p)

  station 0: pi=0.000000, recourse=160.0000
  station 1: pi=43.221098, recourse=0.0000
Iteration 1: LB=1805.0000, UB=1965.0000, gap=160.0000
  station 0: pi=0.000000, recourse=160.0000
  station 1: pi=43.221098, recourse=0.0000
Iteration 2: LB=1965.0000, UB=1965.0000, gap=0.0000
Finished in 2 iterations. Best incumbent (UB) = 1965.0000


(np.float64(1965.0000000103382),
 np.float64(1965.0000000103005),
 array([1., 0.]),
 array([[1., 0.]]),
 array([15.,  0.]),
 [(0, 160.000000009443, 5.715784546221898e-11),
  (1, 3.767897612415051e-11, 43.22109786258486),
  (0, 160.000000009443, 5.715784546221898e-11),
  (1, 3.767897612415051e-11, 43.22109786258486)])